# OpenPOPCON radiative ARC example

An ARC-class, high-field D-T reactor run in a radiative L-mode: 4.2 m,
11.5 T, 14 MA, with krypton seeded to Zeff = 2.1 and the `H89` L-mode
confinement scaling at H = 1.3.

This example uses parabolic profiles rather than a gEQDSK, which keeps `R`,
`a`, `kappa`, `delta` and `I_P` free for `POPCON_scan` to vary.

In [ ]:
import numpy as np
import openpopcon as op

## Setup and run

The numerics are compiled with numba the first time they run, so the first
solve in a fresh kernel takes a few seconds longer than the rest.

In [ ]:
settingsfile = "./POPCON_input_example.yaml"
plotsettingsfile = "./plotsettings.yml"

pc = op.POPCON(settingsfile=settingsfile, plotsettingsfile=plotsettingsfile)
pc.run_POPCON()

## Plotting

The contours are the defaults; this example's plotsettings only move the
axes to on-axis density and temperature.

In [ ]:
fig, ax = pc.plot()

### A point on the plot

The numbers behind one grid point, picked by Greenwald fraction and
volume-averaged ion temperature.

In [ ]:
i = int(np.abs(pc.output.n_G_frac.values - 0.8).argmin())
j = int(np.abs(pc.output.T_i_avg.values - 8.0).argmin())
point = pc.output.isel(n_index=i, T_index=j)

print(f"n/n_G   = {float(point.n_G_frac):.2f}")
print(f"<T_i>   = {float(point.T_i_avg):.1f} keV")
print(f"P_fus   = {float(point.Pfusion):.1f} MW")
print(f"P_aux   = {float(point.Paux):.1f} MW")
print(f"Q       = {float(point.Q):.2f}")
print(f"beta_N  = {float(point.betaN):.2f}")

## Scoping a single operating point

`single_point` solves one density/temperature pair and shows the profiles
behind it, which is the quickest way to see why a point on the POPCON sits
where it does.

In [ ]:
pc.single_point(n_G_frac=0.8, Ti_av=8.0)

## Scanning the seeding against the confinement

`POPCON_scan` runs a full POPCON at every combination of two machine
parameters. The ranges come from the `scan:` block at the bottom of the
settings file: `Zeff_target` on the rows, which re-derives the krypton
fraction in every cell, and `H_fac` on the columns.

In [ ]:
sc = op.POPCON_scan(settingsfile=settingsfile, plotsettingsfile=plotsettingsfile)
sc.run_scan()

In [ ]:
fig, axs = sc.plot()

`plot_metric` reduces each cell to one number so the trend across the scan
reads at a glance. Anything in the output works, with any reduction.

In [ ]:
fig, ax = sc.plot_metric("Q", reduce="max")

The whole scan is also available as one xarray Dataset, with the two scanned
parameters as extra dimensions, for any analysis the plots do not cover.

In [ ]:
sc.output

## Saving and reloading a scan

`write_output` saves the whole scan: the base settings, the scan
specification, the combined arrays and the grid plot.

In [ ]:
sc.write_output(name="radiative_arc_scan", archive=False, overwrite=True)

back = op.POPCON_scan.read_output("radiative_arc_scan")
print(back.shape, back.row.parameter, back.col.parameter)